# User+WC MoE: Decay vs No Decay

Tests whether time-decay weighting helps the User+WC MoE approach.
Compares flat weighting (rate=0) vs decay (rate=0.03) on the winning configuration.

**Dataset:** NLR Kestrel, expanded window (~193 days, 2.7M rows)
**Model:** XGBoost Adjusted (200 trees, depth 12)
**Lookback:** 120 days
**Rolling eval:** 120 windows × 6h

## 1. Setup

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

from hpc_oda_commons.models.experimental.xgboost_adjusted_model import (
    ExperimentalXGBoostAdjustedConfig, ExperimentalXGBoostAdjustedModel,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

REPO_ROOT = Path.cwd().parent.parent
DATA_PATH = REPO_ROOT / 'workspace' / 'data' / 'datasets' / 'nlr_kestrel' / 'data.parquet'

In [ ]:
# Load with enough history for 120-day lookback
table = pq.read_table(DATA_PATH)
lo = datetime(2025, 1, 1, tzinfo=timezone.utc)
hi = datetime(2025, 6, 26, tzinfo=timezone.utc) + timedelta(days=1)
sc = table.column('submit_time')
ec = table.column('end_time')
mask = pc.and_(
    pc.less(sc, pa.scalar(hi, type=sc.type)),
    pc.greater_equal(ec, pa.scalar(lo, type=ec.type)),
)
df = table.filter(mask).to_pandas()
rows_all = df.to_dict('records')
print(f'Loaded: {len(rows_all):,} rows')
print(f'Date range: {df["submit_time"].min().date()} to {df["submit_time"].max().date()}')
span = (df['submit_time'].max() - df['submit_time'].min()).days
print(f'Span: {span} days')

## 2. Configuration

In [ ]:
N_WINDOWS = 120
TEST_WINDOW_HOURS = 6
TRAINING_LOOKBACK_DAYS = 120
DECAY_RATE = 0.03

POWER_USER_PERCENTILE = 0.99

BIN_EDGES_H = [0, 2, 4, 24, 48, float('inf')]
BIN_LABELS = ['<=2h', '2-4h', '4-24h', '24-48h', '>48h']

def make_config(decay_rate):
    return ExperimentalXGBoostAdjustedConfig(
        n_windows=N_WINDOWS,
        test_window_hours=TEST_WINDOW_HOURS,
        training_lookback_days=TRAINING_LOOKBACK_DAYS,
        max_svd_components=64,
        target_max_one_hot_width=512,
        random_state=42,
        n_estimators=200,
        max_depth=12,
        learning_rate=0.03,
        min_child_weight=5,
        gamma=0.1,
        time_decay_rate=decay_rate,
    )

print(f'Windows: {N_WINDOWS} x {TEST_WINDOW_HOURS}h = {N_WINDOWS*TEST_WINDOW_HOURS/24:.0f} days test')
print(f'Lookback: {TRAINING_LOOKBACK_DAYS} days')
print(f'Decay rate: {DECAY_RATE} (day 0=1.0, day 60={np.exp(-DECAY_RATE*60):.2f}, day 120={np.exp(-DECAY_RATE*120):.2f})')

## 3. Helpers

In [ ]:
import time

def assign_bin(row):
    wc_h = (row.get('requested_seconds') or 0) / 3600
    for i in range(len(BIN_EDGES_H) - 1):
        if wc_h <= BIN_EDGES_H[i + 1]:
            return BIN_LABELS[i]
    return BIN_LABELS[-1]


def run_user_wc_moe(rows_all, power_users, decay_rate):
    """Run full User+WC MoE with given decay rate. Returns combined results."""
    all_payloads = []
    
    # Pre-compute all bins to know total count
    bin_jobs = []
    for user in sorted(power_users):
        user_rows = [r for r in rows_all if r.get('user') == user]
        user_bins = {}
        for r in user_rows:
            bl = assign_bin(r)
            user_bins.setdefault(bl, []).append(r)
        for bl in BIN_LABELS:
            bin_rows = user_bins.get(bl, [])
            if len(bin_rows) >= 100:
                bin_jobs.append((f'power {user[:7]}/{bl}', bin_rows))
    
    non_power_rows = [r for r in rows_all if r.get('user') not in power_users]
    np_bins = {}
    for r in non_power_rows:
        bl = assign_bin(r)
        np_bins.setdefault(bl, []).append(r)
    for bl in BIN_LABELS:
        bin_rows = np_bins.get(bl, [])
        if len(bin_rows) >= 100:
            bin_jobs.append((f'non-power/{bl}', bin_rows))
    
    total_bins = len(bin_jobs)
    start_time = time.time()
    
    for idx, (bin_name, bin_rows) in enumerate(bin_jobs):
        elapsed = (time.time() - start_time) / 60
        if idx > 0:
            avg_per_bin = elapsed / idx
            remaining = avg_per_bin * (total_bins - idx)
        else:
            remaining = 0
        
        print(f'  [{idx+1}/{total_bins} bins | {elapsed:.0f}min elapsed | ~{remaining:.0f}min remaining] '
              f'{bin_name} ({len(bin_rows):,} rows)...')
        
        bin_start = time.time()
        try:
            model = ExperimentalXGBoostAdjustedModel(make_config(decay_rate))
            payload = model.evaluate(bin_rows, capture_artifacts=True)
            if payload['summary']['rows_scored'] > 0:
                all_payloads.append(payload)
                bin_elapsed = (time.time() - bin_start) / 60
                print(f'    -> scored={payload["summary"]["rows_scored"]:,}, MAE={payload["mae"]:,.0f}s ({bin_elapsed:.1f}min)')
        except Exception as e:
            print(f'    -> FAILED: {e}')
    
    total_elapsed = (time.time() - start_time) / 60
    print(f'  [DONE | {total_elapsed:.0f}min total]')
    
    # Combine
    all_true = []
    all_pred = []
    for p in all_payloads:
        if '_y_true' in p and '_y_pred' in p:
            all_true.extend(p['_y_true'])
            all_pred.extend(p['_y_pred'])
    
    if not all_true:
        return None
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    mae = np.mean(np.abs(all_true - all_pred))
    rmse = np.sqrt(np.mean((all_true - all_pred)**2))
    return {'mae': mae, 'rmse': rmse, 'scored': len(all_true)}


# Identify power users
user_counts = Counter(r.get('user') for r in rows_all)
threshold = np.percentile(list(user_counts.values()), POWER_USER_PERCENTILE * 100)
power_users = {u for u, c in user_counts.items() if c >= threshold}
power_jobs = sum(1 for r in rows_all if r.get('user') in power_users)
print(f'Power users: {len(power_users)} ({power_jobs:,} jobs, {power_jobs/len(rows_all)*100:.1f}%)')


## 4. Run: Flat weighting (rate=0)

In [ ]:
print('Running User+WC MoE with FLAT weighting (rate=0)...')
result_flat = run_user_wc_moe(rows_all, power_users, 0.0)
if result_flat:
    print(f'\n  FLAT: MAE={result_flat["mae"]:,.0f}s, RMSE={result_flat["rmse"]:,.0f}s, scored={result_flat["scored"]:,}')

## 5. Run: Time-decay weighting (rate=0.03)

In [ ]:
print(f'Running User+WC MoE with DECAY weighting (rate={DECAY_RATE})...')
result_decay = run_user_wc_moe(rows_all, power_users, DECAY_RATE)
if result_decay:
    print(f'\n  DECAY: MAE={result_decay["mae"]:,.0f}s, RMSE={result_decay["rmse"]:,.0f}s, scored={result_decay["scored"]:,}')

## 6. Results

In [ ]:
print('=' * 60)
print('RESULTS: FLAT vs DECAY')
print('=' * 60)

if result_flat and result_decay:
    print(f'\n{"Approach":<30} {"MAE":>10} {"RMSE":>10} {"Scored":>10}')
    print('-' * 65)
    print(f'{"Flat (rate=0)":<30} {result_flat["mae"]:>10,.0f}s {result_flat["rmse"]:>10,.0f}s {result_flat["scored"]:>10,}')
    print(f'{f"Decay (rate={DECAY_RATE})":<30} {result_decay["mae"]:>10,.0f}s {result_decay["rmse"]:>10,.0f}s {result_decay["scored"]:>10,}')
    
    diff = (result_decay['mae'] - result_flat['mae']) / result_flat['mae'] * 100
    print(f'\nDifference: {diff:+.1f}% MAE')
    if diff < -2:
        print('Time-decay HELPS — recent jobs are more informative than older ones.')
    elif diff > 2:
        print('Time-decay HURTS — older jobs are still relevant, weighting them down loses signal.')
    else:
        print('Minimal difference — decay has negligible effect at this lookback length.')

In [ ]:
# Simple bar chart
if result_flat and result_decay:
    fig, ax = plt.subplots(figsize=(7, 4))
    names = ['Flat (rate=0)', f'Decay (rate={DECAY_RATE})']
    maes = [result_flat['mae'], result_decay['mae']]
    bars = ax.bar(names, maes, color=['steelblue', 'coral'], width=0.4)
    for bar, mae in zip(bars, maes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{mae:,.0f}s', ha='center', fontsize=12, fontweight='bold')
    ax.set_ylabel('MAE (seconds)')
    ax.set_title('User+WC MoE: Flat vs Time-Decay Weighting\n(120-day lookback, NLR Kestrel)')
    plt.tight_layout()
    plt.show()

## 7. Per-Bin Comparison Table

Side-by-side comparison of flat vs decay MAE for each bin.
Run this cell AFTER the notebook completes and is saved.

In [ ]:
# Build per-bin comparison from the stored payloads
# (requires result_flat_bins and result_decay_bins to exist from the runs above)
# If they don't exist, we parse from the printed output

import re
from pathlib import Path

nb_file = Path.cwd() / 'user_wc_moe_decay_comparison.ipynb'
nb_data = json.loads(nb_file.read_text())

flat_bins = {}
decay_bins = {}
current_run = None

for cell in nb_data['cells']:
    if cell['cell_type'] != 'code' or not cell.get('outputs'):
        continue
    for output in cell['outputs']:
        if not output.get('text'):
            continue
        text = ''.join(output['text'])
        if 'FLAT weighting' in text:
            current_run = 'flat'
        elif 'DECAY weighting' in text:
            current_run = 'decay'
        for line in text.split('\n'):
            m = re.search(r'scored=([\d,]+),\s*MAE=([\d,]+)s', line)
            if m and current_run:
                # Extract bin name from the line
                bm = re.search(r'((?:power|non-power)\S+)', line)
                if bm:
                    bin_name = bm.group(1)
                    scored = int(m.group(1).replace(',', ''))
                    mae = int(m.group(2).replace(',', ''))
                    if current_run == 'flat':
                        flat_bins[bin_name] = {'scored': scored, 'mae': mae}
                    else:
                        decay_bins[bin_name] = {'scored': scored, 'mae': mae}

print(f'Flat bins: {len(flat_bins)}, Decay bins: {len(decay_bins)}')
print()
print(f'{"Bin":<35} {"Scored":>8} {"Flat MAE":>10} {"Decay MAE":>10} {"Change":>10}')
print('-' * 78)

for bin_name in sorted(flat_bins.keys(), key=lambda b: -flat_bins[b]['scored']):
    f = flat_bins[bin_name]
    d = decay_bins.get(bin_name)
    if d:
        change = (d['mae'] - f['mae']) / f['mae'] * 100 if f['mae'] > 0 else 0
        marker = ' <<<' if change < -10 else (' >>>' if change > 10 else '')
        print(f'{bin_name:<35} {f["scored"]:>8,} {f["mae"]:>10,}s {d["mae"]:>10,}s {change:>+9.1f}%{marker}')
    else:
        print(f'{bin_name:<35} {f["scored"]:>8,} {f["mae"]:>10,}s {"(missing)":>10}')
